# Expected Returns Analytics
# 
# This notebook analyzes expected returns using the enhanced v2.0 analytics pipeline:
# - **Monte Carlo Simulation** — Probabilistic upside/downside distributions
# - **Price Target Achievement** — Probability-weighted expected returns by sector
# - **Kalman Filtered Targets** — Noise-reduced price target signals
# - **Analyst Sentiment Features** — Feature-level probability analytics
# - **Cross-Model Comparison** — MC vs Kalman vs Achievement model alignment
#
# Data sources: `analytics.monte_carlo_simulation`, `analytics.price_target_achievement`,
# `analytics.kalman_filtered_price_targets`, `analytics.earnings_probability_analysis`


## 1. Setup & Environment Configuration


In [1]:
import warnings
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings("ignore")

# Configure database connection
if "DB_URL" not in os.environ:
    env_file = "environment_variables.txt"
    if os.path.exists(env_file):
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ[key.strip()] = value.strip()

PLOTLY_TEMPLATE = "plotly_dark"
COLORS = px.colors.qualitative.Dark24

print("✅ Environment configured")

# ── Refactored analytics modules ─────────────────────────────────────
from finance_ml.analytics.data_utils import (
    export_to_analytics_db,
    load_identifier_columns,
)
from finance_ml.analytics.probability_analytics import (
    PriceTargetAchievementModel,
    EarningsBeatProbabilityModel,
    EPSStreakAnalyzer,
    CreditRiskProbabilityModel,
    create_earnings_probability_dashboard,
)
from finance_ml.analytics.statistical_analysis import (
    kalman_filter_price_target,
    kalman_momentum_filter,
    fit_gaussian_copula,
)
from finance_ml.analytics.optimized_ops import (
    fast_monte_carlo_simulation,
    get_optimization_status,
)
from finance_ml.analytics.visualizations import (
    create_analyst_upside_scatter,
    create_valuation_vs_growth_quadrant,
)

# --- InferenceData schema (ArviZ / xarray bridge) ---
try:
    from finance_ml.analytics.inference_schema import (
        ARVIZ_AVAILABLE,
        build_monte_carlo_inference_data,
        summarize_inference_data,
    )
except ImportError:
    ARVIZ_AVAILABLE = False

# --- Probabilistic visualizations (ArviZ-backed) ---
from finance_ml.analytics.visualizations.probability_viz import (
    create_posterior_return_forest,
    create_beat_probability_posterior,
    create_ruin_probability_diagnostic,
    create_bayesian_category_ridge,
    create_tri_model_posterior_comparison,
)


✅ Environment configured


## 2. Data Acquisition


In [2]:
%%sql
SELECT * FROM analytics.monte_carlo_simulation

,next_earnings_status,dividend_record_record_date,style_class,industry,next_earnings,next_earnings_when,exchange,size_class,dividend_record_payable_date,name,...,price_target_median,shares_outstanding,pt_median,pt_spread,expected_upside_pct,upside_std,var_5_pct,prob_positive_upside,risk_reward_ratio,volume_shrs
0,Confirmed,2025-03-31,Core,Machinery,2026-02-13,Pre-Market,HLSE,Mid Cap,2025-04-07,Kalmar Oyj,...,48.0000,64142729,48.0000,13.0000,4.956073,5.741032,-4.380980,79.05,0.863272,NaN
1,Confirmed,2025-05-08,Core,Machinery,2026-02-13,Pre-Market,OB,Mid Cap,2025-05-20,Tomra Systems ASA,...,150.3479,295395838,150.3479,88.5528,24.795381,14.810019,0.475844,95.37,1.674230,783902.0
2,Estimated,2025-09-29,Core,Consumer Staples Distribution and Retail,2026-02-13,Pre-Market,ENXTBR,Mid Cap,2025-09-30,Colruyt Group N.V.,...,37.5000,119780970,37.5000,19.0600,7.396648,11.060413,-11.106130,73.84,0.668750,17742.0
3,Confirmed,2025-04-23,Core,Construction and Engineering,2026-02-13,Pre-Market,ENXTAM,Mid Cap,2025-04-29,Koninklijke Heijmans N.V.,...,80.2500,27478005,80.2500,23.5000,-8.573839,5.471292,-17.774274,6.34,-1.567059,228576.0
4,Estimated,NaN,Core,Diversified Consumer Services,2026-02-13,Pre-Market,LSE,Small Cap,NaN,Auction Technology Group plc,...,4.6000,121108423,4.6000,5.0500,75.790774,35.321471,24.002795,100.00,2.145742,375918.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5373,Estimated,2025-11-13,Core,Food Products,2026-05-15,During-Market,NSEI,Mid Cap,2025-12-07,Patanjali Foods Limited,...,677.5000,1087918113,677.5000,100.0000,25.341502,4.109957,17.776877,100.00,6.165880,1260588.0
5374,Estimated,2025-08-01,Growth,Automobile Components,2026-05-15,During-Market,NSEI,Mid Cap,2025-08-18,Endurance Technologies Limited,...,2976.0000,140662848,2976.0000,810.0000,18.723600,6.705077,7.257897,100.00,2.792451,64637.0
5375,Estimated,2025-08-28,Value,Beverages,2026-05-15,After-Market,ASX,Mid Cap,2025-10-02,Treasury Wine Estates Limited,...,5.2000,807439901,5.2000,3.7500,23.976026,16.947565,1.638678,97.50,1.414718,12247400.0
5376,Estimated,2026-03-31,Value,Transportation Infrastructure,2026-05-15,After-Market,TSE,Mid Cap,2026-06-30,Kamigumi Co. Ltd.,...,5500.0000,99384031,5500.0000,900.0000,-2.830148,3.349982,-8.926641,21.60,-0.844825,464300.0


In [3]:
%%sql
SELECT * FROM analytics.price_target_achievement

,isin,ticker,name,region,country,trading_country,exchange,sector,industry,dividend_record_frequency,...,next_income_statement_report_date,reference_date,achievement_probability,upside_potential,price_target_spread_pct,analyst_conviction,eps_revision_momentum,analyst_rating_normalized,expected_return_prob_weighted,confidence_level
0,FI4000571054,KALMAR,Kalmar Oyj,Europe,FI,FI,HLSE,Industrials,Machinery,Annual,...,2026-06-30,2026-02-17,0.78,4.347826,27.083333,60.000000,0.045175,75.00,3.391304,Medium
1,NO0012470089,TOM,Tomra Systems ASA,Europe,NO,NO,OB,Industrials,Machinery,Annual,...,2026-06-30,2026-02-17,0.35,22.433143,58.898595,62.500000,-0.023120,75.00,7.851600,Low
2,BE0974256852,COLR,Colruyt Group N.V.,Europe,BE,BE,ENXTBR,Consumer Staples,Consumer Staples Distribution and Retail,Annual,...,2026-03-30,2026-02-17,0.66,7.449857,50.826667,11.111111,0.015800,61.00,4.916905,Low
3,SE0017615784,SKOLON,Skolon AB (publ),Europe,SE,SE,OM,Information Technology,Software,NaN,...,2025-12-30,2026-02-17,0.28,47.058824,0.000000,0.000000,0.000000,-25.00,13.176471,Low
4,SE0010442418,BAHNB,Bahnhof AB (publ),Europe,SE,SE,OM,Communication Services,Diversified Telecommunication Services,Annual,...,2025-12-30,2026-02-17,0.40,30.597015,0.000000,100.000000,0.039265,75.00,12.238806,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6169,INE619A01035,PATANJALI,Patanjali Foods Limited,Asia / Pacific,IN,IN,NSEI,Consumer Staples,Food Products,Interim Payment,...,2026-03-31,2026-02-17,0.60,28.875785,14.760148,100.000000,0.000000,93.75,17.325471,High
6170,INE913H01037,ENDURANCE,Endurance Technologies Limited,Asia / Pacific,IN,IN,NSEI,Consumer Discretionary,Automobile Components,Annual,...,2026-03-31,2026-02-17,0.68,19.590115,27.217742,60.000000,-0.044975,76.75,13.321278,Medium
6171,AU000000TWE9,TWE,Treasury Wine Estates Limited,Asia / Pacific,AU,AU,ASX,Consumer Staples,Beverages,Final Payment,...,2026-06-30,2026-02-17,0.68,4.627767,72.115385,6.666667,-0.186360,51.75,3.146881,Low
6172,JP3219000001,9364,Kamigumi Co. Ltd.,Asia / Pacific,JP,JP,TSE,Industrials,Transportation Infrastructure,Final Payment,...,2026-03-31,2026-02-17,0.85,-1.025733,16.363636,66.666667,0.000000,75.00,-0.871873,High


In [4]:
%%sql
SELECT * FROM analytics.kalman_filtered_price_targets

,ticker,name,country,exchange,sector,industry,kalman_estimate,kalman_variance,kalman_gain,signal_strength,original_price,original_target,filtered_upside
0,KALMAR,Kalmar Oyj,FI,HLSE,Industrials,Machinery,47.818183,0.090909,0.909092,10.99999,46.00,48.0000,3.952573
1,TOM,Tomra Systems ASA,NO,OB,Industrials,Machinery,147.843568,0.090909,0.909092,10.99999,122.80,150.3479,20.393785
2,COLR,Colruyt Group N.V.,BE,ENXTBR,Consumer Staples,Consumer Staples Distribution and Retail,37.263639,0.090909,0.909092,10.99999,34.90,37.5000,6.772603
3,SKOLON,Skolon AB (publ),SE,OM,Information Technology,Software,36.409101,0.090909,0.909092,10.99999,25.50,37.5000,42.780788
4,BAHNB,Bahnhof AB (publ),SE,OM,Communication Services,Diversified Telecommunication Services,68.509104,0.090909,0.909092,10.99999,53.60,70.0000,27.815493
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6166,PATANJALI,Patanjali Foods Limited,IN,NSEI,Consumer Staples,Food Products,663.700125,0.090909,0.909092,10.99999,525.70,677.5000,26.250737
6167,ENDURANCE,Endurance Technologies Limited,IN,NSEI,Consumer Discretionary,Automobile Components,2931.682221,0.090909,0.909092,10.99999,2488.50,2976.0000,17.809211
6168,TWE,Treasury Wine Estates Limited,AU,ASX,Consumer Staples,Beverages,5.179091,0.090909,0.909092,10.99999,4.97,5.2000,4.207064
6169,9364,Kamigumi Co. Ltd.,JP,TSE,Industrials,Transportation Infrastructure,5505.181771,0.090909,0.909092,10.99999,5557.00,5500.0000,-0.932486


In [5]:
%%sql
SELECT * FROM public.vw_features_analyst_sentiment
ORDER BY next_earnings ASC

,isin,ticker,name,region,country,trading_country,exchange,sector,industry,dividend_record_frequency,...,pt_median_momentum_1m,pt_median_momentum_3m,pt_acceleration_short,pt_acceleration_long,pt_consensus_convergence,analyst_coverage_change_1m,analyst_coverage_change_3m,analyst_coverage_change_1y,pt_vs_price_momentum,analyst_coverage_trend
0,SE0000407991,SVEDB,Svedbergs Group AB (publ),Europe,SE,SE,OM,Industrials,Building Products,Interim Payment,...,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN
1,GB00BMVQDZ64,ATG,Auction Technology Group plc,Europe,GB,GB,LSE,Consumer Discretionary,Diversified Consumer Services,NaN,...,0.039548,-0.215686,0.305861,0.074093,-0.356138,2,0,0,-0.216676,0.100000
2,BE0974256852,COLR,Colruyt Group N.V.,Europe,BE,BE,ENXTBR,Consumer Staples,Consumer Staples Distribution and Retail,Annual,...,0.020408,-0.107143,0.089068,0.045338,-0.070171,1,1,0,-0.175481,0.111111
3,SE0017615784,SKOLON,Skolon AB (publ),Europe,SE,SE,OM,Information Technology,Software,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0,0.019608,0.000000
4,NL0009269109,HEIJM,Koninklijke Heijmans N.V.,Europe,NL,NL,ENXTAM,Industrials,Construction and Engineering,Annual,...,0.140320,0.140320,0.000000,-1.228684,-0.239549,0,0,0,-0.248883,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6411,AU000000COH5,COH,Cochlear Limited,Asia / Pacific,AU,AU,ASX,Health Care,Health Care Equipment and Supplies,Interim Payment,...,-0.182412,-0.182413,0.006303,-0.024113,-0.023560,0,0,0,0.150567,-0.016667
6412,NL0015000D50,NXFIL,NX Filtration N.V.,Europe,NL,NL,ENXTAM,Industrials,Machinery,NaN,...,-0.140845,-0.197368,0.123520,0.056311,-0.221311,0,0,-1,0.011842,-0.062500
6413,AT000000STR1,STR,Strabag SE,Europe,AT,AT,WBAG,Industrials,Construction and Engineering,Annual,...,0.076796,0.076796,0.000000,-0.805274,0.032984,0,0,0,-0.199618,0.000000
6414,FR0004027068,ALLAN,Lanson-BCC,Europe,FR,FR,ENXTPA,Consumer Staples,Beverages,Annual,...,-0.048193,-0.048193,0.000000,0.022395,-0.109806,0,0,0,0.054015,0.000000


## 3. Data Overview & Quality Checks


In [6]:
# Rename the DataSpell-imported variables to convenient names
# (Adjust variable names if DataSpell assigns different ones)
try:
    mc = mc_sim.copy()
except NameError:
    print("⚠️ Run the data_input cells above first")

try:
    pt = pt_a.copy()
except NameError:
    print("⚠️ Run the price_target_achievement data_input cell first")

try:
    kal = pt_kal.copy()
except NameError:
    print("⚠️ Run the kalman_filtered_price_targets data_input cell first")

print(f"Monte Carlo Simulation:        {mc.shape[0]:,} stocks × {mc.shape[1]} cols")
print(f"Price Target Achievement:      {pt.shape[0]:,} stocks × {pt.shape[1]} cols")
print(f"Kalman Filtered Targets:       {kal.shape[0]:,} stocks × {kal.shape[1]} cols")

# Summary statistics for core return metrics
display(mc[["expected_upside_pct", "var_5_pct", "prob_positive_upside", "risk_reward_ratio"]].describe().round(2))


Monte Carlo Simulation:        5,378 stocks × 47 cols
Price Target Achievement:      6,174 stocks × 39 cols
Kalman Filtered Targets:       6,171 stocks × 13 cols


,expected_upside_pct,var_5_pct,prob_positive_upside,risk_reward_ratio
count,5378.00,5378.00,5378.00,5378.00
mean,22.89,3.51,75.07,2.81
std,40.05,30.73,32.57,12.02
min,-73.78,-79.00,0.00,-164.72
25%,1.15,-14.50,55.40,0.13
50%,13.84,-0.63,94.00,1.54
75%,33.27,16.15,100.00,3.32
max,670.34,322.93,100.00,533.49


## 4. Monte Carlo Simulation Analysis


### 4.1 Expected Upside Distribution


In [7]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Expected Upside Distribution", "Probability of Positive Return"),
    vertical_spacing=0.12,
)

# Clip extreme outliers for better visualization
upside_clipped = mc["expected_upside_pct"].clip(-100, 300)

fig.add_trace(
    go.Histogram(
        x=upside_clipped,
        nbinsx=80,
        marker_color=COLORS[0],
        opacity=0.75,
        name="Expected Upside %",
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=1)
fig.add_vline(
    x=mc["expected_upside_pct"].median(),
    line_dash="dot", line_color="green",
    annotation_text=f"Median: {mc['expected_upside_pct'].median():.1f}%",
    row=1, col=1,
)

# Probability of positive return - pie chart
prob_bins = pd.cut(mc["prob_positive_upside"], bins=[0, 25, 50, 75, 100],
                   labels=["0-25%", "25-50%", "50-75%", "75-100%"])
prob_counts = prob_bins.value_counts().sort_index()
fig.add_trace(
    go.Bar(
        x=prob_counts.index.astype(str),
        y=prob_counts.values,
        marker_color=[COLORS[3], COLORS[1], COLORS[0], COLORS[2]],
        name="Stock Count",
    ),
    row=2, col=1,
)

fig.update_layout(
    title="Monte Carlo Simulation: Return Distribution Overview",
    template=PLOTLY_TEMPLATE,
    height=800,
    width=1000,
    showlegend=True,
)
fig.update_xaxes(title_text="Expected Upside (%)", row=1, col=1)
fig.update_xaxes(title_text="Probability of Positive Return", row=2, col=1)
fig.update_yaxes(title_text="Number of Stocks", row=1, col=1)
fig.update_yaxes(title_text="Number of Stocks", row=2, col=1)
fig.show()


### 4.2 Risk-Reward by Industry


In [8]:
# Sector-level aggregation
mc_sector = (
    mc.groupby("industry")
    .agg(
        mean_upside=("expected_upside_pct", "mean"),
        median_upside=("expected_upside_pct", "median"),
        mean_var5=("var_5_pct", "mean"),
        mean_prob_positive=("prob_positive_upside", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_upside", ascending=False)
)

fig = px.scatter(
    mc_sector,
    x="mean_var5",
    y="mean_upside",
    size="count",
    color="industry",
    hover_name="industry",
    hover_data={"mean_prob_positive": ":.1f", "count": True},
    title="Industry Risk-Reward: Expected Upside vs Value-at-Risk (5%)",
    labels={
        "mean_var5": "Mean VaR 5% (%)",
        "mean_upside": "Mean Expected Upside (%)",
        "count": "# Stocks",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.show()


### 4.3 Top Opportunities — Highest Risk-Reward Ratio (Positive Upside)


In [9]:
mc_positive = mc[mc["prob_positive_upside"] >= 75].nlargest(50, "risk_reward_ratio")

fig = px.bar(
    mc_positive,
    x="ticker",
    y="expected_upside_pct",
    color="industry",
    hover_data=["name", "prob_positive_upside", "risk_reward_ratio"],
    title="Top 50 Opportunities: Highest Risk-Reward (≥75% Prob Positive)",
    labels={"expected_upside_pct": "Expected Upside (%)", "ticker": "Ticker"},
    template=PLOTLY_TEMPLATE,
    height=500,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


## 5. Price Target Achievement Analysis


### 5.1 Achievement Probability Distribution by Confidence Level


In [10]:
fig = px.violin(
    pt,
    x="confidence_level",
    y="achievement_probability",
    color="confidence_level",
    box=True,
    points="outliers",
    title="Price Target Achievement Probability by Confidence Level",
    labels={
        "achievement_probability": "Achievement Probability",
        "confidence_level": "Confidence Level",
    },
    category_orders={"confidence_level": ["Low", "Medium", "High"]},
    color_discrete_sequence=[COLORS[3], COLORS[1], COLORS[2]],
    template=PLOTLY_TEMPLATE,
    height=450,
)
fig.show()


### 5.2 Probability-Weighted Expected Return by Sector


In [11]:
pt_sector = (
    pt.groupby("industry")
    .agg(
        mean_expected_return=("expected_return_prob_weighted", "mean"),
        median_expected_return=("expected_return_prob_weighted", "median"),
        mean_achievement_prob=("achievement_probability", "mean"),
        mean_conviction=("analyst_conviction", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_expected_return", ascending=True)
)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=pt_sector["industry"],
        x=pt_sector["mean_expected_return"],
        orientation="h",
        marker_color=[
            COLORS[2] if v >= 0 else COLORS[3]
            for v in pt_sector["mean_expected_return"]
        ],
        text=pt_sector["mean_expected_return"].apply(lambda v: f"{v:.1f}%"),
        textposition="outside",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Mean Prob-Weighted Return: %{x:.2f}%<br>"
            "Avg Achievement Prob: %{customdata[0]:.0%}<br>"
            "Avg Conviction: %{customdata[1]:.1f}<br>"
            "Stocks: %{customdata[2]}"
        ),
        customdata=pt_sector[["mean_achievement_prob", "mean_conviction", "count"]].values,
    )
)
fig.update_layout(
    title="Probability-Weighted Expected Return by Industry",
    xaxis_title="Mean Expected Return (%)",
    template=PLOTLY_TEMPLATE,
    height=1100,
    margin=dict(l=350),
)
fig.show()


### 5.3 Conviction vs Upside Potential Scatter


In [12]:
sample_pt = pt.dropna(subset=["analyst_conviction"]).sample(min(2000, len(pt)), random_state=42)
fig = px.scatter(
    sample_pt,
    x="expected_return_prob_weighted",
    y="upside_potential",
    color="achievement_probability",
    size="analyst_conviction",
    hover_name="ticker",
    hover_data=["name", "sector", "industry", "expected_return_prob_weighted", "confidence_level"],
    title="Analyst Conviction vs Upside Potential",
    labels={
        "analyst_conviction": "Analyst Conviction (%)",
        "upside_potential": "Upside Potential (%)",
    },
    category_orders={"confidence_level": ["Low", "Medium", "High"]},
    color_discrete_sequence=[COLORS[3], COLORS[1], COLORS[2]],
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.6,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.4)
fig.show()


## 5.5 Earnings Beat Probability × Price Target Alignment


In [13]:
# Cross-reference: stocks with high beat probability AND high achievement probability
beat_model = EarningsBeatProbabilityModel()
_beat_source_df = beat_results  # the loaded equities DataFrame from a prior cell
beat_results = beat_model.analyze_dataframe_enhanced(
    # analyst_sentiment has forward EPS estimates, revision momentum, and reported EPS history
    # that the three-layer fusion model requires
    _beat_source_df,
    sector_col='sector' if 'sector' in _beat_source_df.columns else 'industry',
    ticker_col='ticker'
)

if len(beat_results) > 0 and len(analyst_sentiment) > 0:
    beat_pt = beat_results[['ticker', 'posterior_beat_prob', 'confidence_score']].merge(
        pt[['ticker', 'achievement_probability', 'expected_return_prob_weighted']],
        on='ticker', how='inner',
    )
    
    fig = px.scatter(
        beat_pt,
        x='posterior_beat_prob',
        y='achievement_probability',
        color='expected_return_prob_weighted',
        hover_name='ticker',
        title='Earnings Beat Probability vs Price Target Achievement',
        labels={
            'posterior_beat_prob': 'P(Beat Next Quarter)',
            'achievement_probability': 'P(Reach Price Target)',
        },
        color_continuous_scale='RdYlGn',
        template=PLOTLY_TEMPLATE,
        height=500,
    )
    fig.show()

    # Dual-signal picks: high on both dimensions
    dual_signal = beat_pt[
        (beat_pt['posterior_beat_prob'] > 0.6) &
        (beat_pt['achievement_probability'] > 0.6)
    ].nlargest(30, 'expected_return_prob_weighted')

    print(f"🎯 Dual-signal picks (high beat + high achievement): {len(dual_signal)}")
    display(dual_signal)
else:
    print("⚠️ Insufficient data for beat probability × price target alignment")


⚠️ Insufficient data for beat probability × price target alignment


## 6. Kalman Filtered Price Target Analysis


### 6.1 Kalman Filtered vs Original Upside


In [14]:
# Pre-compute the column on the full DataFrame
kal["raw_upside"] = (kal["original_target"] - kal["original_price"]) / kal["original_price"] * 100

# Sample AFTER the column exists
kal_sample = kal.sample(min(2000, len(kal)), random_state=42).copy()

# Apply signed log1p transform for axis-aligned visualization
kal_sample["filtered_upside_log"] = np.sign(kal_sample["filtered_upside"]) * np.log1p(
    np.abs(kal_sample["filtered_upside"]))
kal_sample["raw_upside_log"] = np.sign(kal_sample["raw_upside"]) * np.log1p(np.abs(kal_sample["raw_upside"]))

fig = px.scatter(
    kal_sample,
    x="filtered_upside_log",
    y="raw_upside_log",
    color="industry",
    hover_name="ticker",
    hover_data=["name", "kalman_estimate", "original_target", "original_price", "filtered_upside", "raw_upside"],
    title="Kalman-Filtered Upside vs Raw Analyst Upside (Log-Transformed Axes)",
    labels={
        "filtered_upside_log": "Kalman Filtered Upside — sign(x)·log₁ₚ(|x|)",
        "raw_upside_log": "Raw Analyst Upside — sign(x)·log₁ₚ(|x|)",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.6,
)

# Add diagonal reference line on the log-transformed scale
log_max = max(
    kal_sample["filtered_upside_log"].abs().quantile(0.99),
    kal_sample["raw_upside_log"].abs().quantile(0.99),
)
fig.add_shape(
    type="line", x0=-log_max, y0=-log_max, x1=log_max, y1=log_max,
    line=dict(color="gray", dash="dash", width=1),
)
fig.show()


### 6.2 Signal Strength Distribution by Sector


In [15]:
fig = px.box(
    kal,
    x="industry",
    y="filtered_upside",
    color="industry",
    title="Kalman-Filtered Upside Distribution by Sector",
    labels={
        "filtered_upside": "Filtered Upside (%)",
        "industry": "",
    },
    template=PLOTLY_TEMPLATE,
    height=1000,
)
fig.update_layout(
    xaxis_tickangle=-65,
    showlegend=False,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5)
fig.show()


### 6.3 Kalman Noise Reduction Effectiveness


In [16]:
kal["raw_upside"] = (kal["original_target"] - kal["original_price"]) / kal["original_price"] * 100
kal["noise_reduction"] = abs(kal["raw_upside"] - kal["filtered_upside"])

noise_by_sector = (
    kal.groupby("industry")
    .agg(
        mean_noise_reduction=("noise_reduction", "mean"),
        median_raw_upside=("raw_upside", "median"),
        median_filtered_upside=("filtered_upside", "median"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_noise_reduction", ascending=False)
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=noise_by_sector["industry"],
    y=noise_by_sector["median_raw_upside"],
    name="Raw Median Upside",
    marker_color=COLORS[1],
    opacity=0.7,
))
fig.add_trace(go.Bar(
    x=noise_by_sector["industry"],
    y=noise_by_sector["median_filtered_upside"],
    name="Kalman-Filtered Median Upside",
    marker_color=COLORS[0],
))
fig.update_layout(
    title="Kalman Filter Impact: Raw vs Filtered Median Upside by Sector",
    yaxis_title="Median Upside (%)",
    barmode="group",
    template=PLOTLY_TEMPLATE,
    height=1000,
    xaxis_tickangle=-85,
)
fig.show()


### 6.4 Export Kalman-Filtered Targets


In [17]:
# Regenerate Kalman-filtered targets using the refactored module
kalman_filtered_price_targets = kalman_filter_price_target(kal)

if len(kalman_filtered_price_targets) > 0:
    export_to_analytics_db(
        _reorder_with_identifiers(kalman_filtered_price_targets),
        "kalman_filtered_price_targets"
    )
    print(f"✓ Exported {len(kalman_filtered_price_targets)} rows → analytics.kalman_filtered_price_targets")


## 7. Cross-Model Comparison


### 7.1 MC Expected Upside vs Kalman Filtered Upside


In [18]:
# Merge Monte Carlo and Kalman results
mc_kal = mc.merge(
    kal[["ticker", "country", "exchange", "filtered_upside", "kalman_estimate", "original_price", "original_target"]],
    on="ticker",
    how="inner",
)

fig = px.scatter(
    mc_kal.sample(min(2000, len(mc_kal)), random_state=42),
    x="expected_upside_pct",
    y="filtered_upside",
    color="industry",
    hover_name="ticker",
    hover_data=["name", "original_price", "kalman_estimate", "original_target", "prob_positive_upside"],
    title="Monte Carlo vs Kalman-Filtered Expected Returns",
    labels={
        "expected_upside_pct": "MC Expected Upside (%)",
        "filtered_upside": "Kalman Filtered Upside (%)",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.5,
)
# Diagonal reference
fig.add_shape(
    type="line", x0=-100, y0=-100, x1=200, y1=200,
    line=dict(color="red", dash="dash", width=2),
)
fig.show()

In [19]:

# Correlation summary
corr = mc_kal[["expected_upside_pct", "filtered_upside"]].corr().iloc[0, 1]
print(f"📊 MC ↔ Kalman correlation: {corr:.3f}")


📊 MC ↔ Kalman correlation: 0.922


### 7.2 Tri-Model Alignment: MC + Kalman + Achievement


In [20]:
# Merge all three models
tri = (
    mc[["ticker", "name", "sector", "industry", "expected_upside_pct", "prob_positive_upside"]]
    .merge(
        kal[["ticker", "filtered_upside"]],
        on="ticker",
        how="inner",
    )
    .merge(
        pt[["ticker", "expected_return_prob_weighted", "achievement_probability", "confidence_level"]],
        on="ticker",
        how="inner",
    )
)

# Agreement score: all three models agree on direction
tri["mc_bullish"] = tri["expected_upside_pct"] > 0
tri["kal_bullish"] = tri["filtered_upside"] > 0
tri["pt_bullish"] = tri["expected_return_prob_weighted"] > 0
tri["agreement_score"] = (
        tri["mc_bullish"].astype(int)
        + tri["kal_bullish"].astype(int)
        + tri["pt_bullish"].astype(int)
)
tri["signal"] = tri["agreement_score"].map(
    {0: "Strong Bearish (0/3)", 1: "Bearish (1/3)", 2: "Bullish (2/3)", 3: "Strong Bullish (3/3)"}
)

fig = px.histogram(
    tri,
    x="signal",
    color="signal",
    title="Tri-Model Signal Agreement (MC + Kalman + Achievement)",
    labels={"signal": "Model Agreement", "count": "Number of Stocks"},
    color_discrete_map={
        "Strong Bearish (0/3)": COLORS[3],
        "Bearish (1/3)": COLORS[1],
        "Bullish (2/3)": COLORS[0],
        "Strong Bullish (3/3)": COLORS[2],
    },
    category_orders={"signal": [
        "Strong Bearish (0/3)", "Bearish (1/3)",
        "Bullish (2/3)", "Strong Bullish (3/3)",
    ]},
    template=PLOTLY_TEMPLATE,
    height=420,
)
fig.update_layout(showlegend=False)
fig.show()

In [21]:

print(f"\n📊 Model Agreement Summary:")
print(tri["signal"].value_counts().to_string())



📊 Model Agreement Summary:
signal
Strong Bullish (3/3)    3958
Strong Bearish (0/3)     991
Bullish (2/3)            249
Bearish (1/3)            180


### 7.3 Strong Consensus Picks — All 3 Models Bullish, High Confidence


In [22]:
strong_consensus = (
    tri[
        (tri["agreement_score"] == 3)
        & (tri["prob_positive_upside"] >= 55)
        & (tri["achievement_probability"] >= 0.6)
        ]
    .nlargest(50, "expected_upside_pct")
)

if len(strong_consensus) > 0:
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["expected_upside_pct"],
        name="MC Expected Upside",
        marker_color=COLORS[0],
    ))
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["filtered_upside"],
        name="Kalman Filtered Upside",
        marker_color=COLORS[1],
    ))
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["expected_return_prob_weighted"],
        name="Prob-Weighted Return",
        marker_color=COLORS[2],
    ))
    fig.update_layout(
        title=f"Top {len(strong_consensus)} Strong Consensus Picks (All 3 Models Bullish)",
        yaxis_title="Expected Return (%)",
        barmode="group",
        template=PLOTLY_TEMPLATE,
        height=500,
        xaxis_tickangle=-45,
    )
    fig.show()

    display(
        strong_consensus[["ticker", "name", "sector", "industry", "expected_upside_pct",
                          "filtered_upside", "expected_return_prob_weighted",
                          "prob_positive_upside", "achievement_probability", "confidence_level"]]
        .reset_index(drop=True)
    )
else:
    print("No stocks meet the strong consensus criteria.")


,ticker,name,sector,industry,expected_upside_pct,filtered_upside,expected_return_prob_weighted,prob_positive_upside,achievement_probability,confidence_level
0,TLEVISACPO,Grupo Televisa S.A.B.,Communication Services,Diversified Telecommunication Services,122.977453,4.200685,3.095902,96.89,0.67,Low
1,SFOR,S4 Capital plc,Consumer Staples,Media,84.855838,13.215871,8.867841,97.94,0.61,Low
2,ARA,Consorcio ARA S. A. B. de C. V.,Consumer Discretionary,Household Durables,76.576017,1.350136,0.920792,98.12,0.62,Low
3,BHIA3,Grupo Casas Bahia S.A.,Consumer Discretionary,Specialty Retail,65.371624,5.274594,3.945392,95.04,0.68,Low
4,ORBIA,Orbia Advance Corporation S.A.B. de C.V.,Materials,Chemicals,48.833297,7.213985,5.792824,85.15,0.73,Low
5,CPB,The Campbell's Company,Consumer Staples,Food Products,41.636595,9.009317,6.243451,98.67,0.63,Low
6,DFS,DFS Furniture plc,Consumer Discretionary,Specialty Retail,40.545814,26.962563,19.278215,100.00,0.65,Low
7,STOK,Stoke Therapeutics Inc.,Health Care,Biotechnology,38.931498,13.175368,10.145024,100.00,0.70,Low
8,XAR,Xaar plc,Information Technology,Technology Hardware Storage and Peripherals,37.933481,26.645792,17.879310,100.00,0.61,Medium
9,CSNA3,Companhia Siderúrgica Nacional,Materials,Metals and Mining,37.659891,8.018335,5.997709,88.13,0.68,Low


### 7.4 Quad-Model Alignment: MC + Kalman + Achievement + EPS Streak


In [23]:
# Quad-model agreement (4/4): MC + Kalman + PT Achievement + Beat Probability
BEAT_BULLISH_THRESHOLD = 0.6

_has_beat_data = (
        not beat_results.empty
        and 'ticker' in beat_results.columns
        and 'posterior_beat_prob' in beat_results.columns
)

if _has_beat_data and 'ticker' in tri.columns:
    beat_slim = beat_results[['ticker', 'posterior_beat_prob']].rename(
        columns={'posterior_beat_prob': 'beat_prob'}
    )
    quad = tri.merge(beat_slim, on='ticker', how='inner')

    if quad.empty:
        print("⚠️ No overlapping tickers between tri-model and beat_results")
    else:
        quad['beat_bullish_flag'] = (quad['beat_prob'] >= BEAT_BULLISH_THRESHOLD).astype(int)
        quad['quad_agreement'] = (
                quad['mc_bullish'].astype(int)
                + quad['kalman_bullish'].astype(int)
                + quad['pt_bullish'].astype(int)
                + quad['beat_bullish_flag']
        )

        fig = px.histogram(
            quad,
            x='quad_agreement',
            nbins=5,
            title='📊 Quad-Model Agreement Distribution (MC + Kalman + PT + Beat)',
            labels={'quad_agreement': 'Models Agreeing (out of 4)'},
            color_discrete_sequence=['#2E91E5'],
            template=PLOTLY_TEMPLATE,
            height=420,
        )
        fig.update_xaxes(dtick=1)
        fig.show()

        full_consensus = (quad['quad_agreement'] == 4).sum()
        no_consensus = (quad['quad_agreement'] == 0).sum()
        print(f"📊 Full consensus (4/4): {full_consensus} stocks")
        print(f"📊 No consensus  (0/4): {no_consensus} stocks")
        print(f"📊 Total quad-model coverage: {len(quad):,} stocks")
else:
    reasons = []
    if beat_results.empty:
        reasons.append("beat_results is empty")
    elif 'ticker' not in beat_results.columns:
        reasons.append("beat_results missing 'ticker' column")
    elif 'posterior_beat_prob' not in beat_results.columns:
        reasons.append("beat_results missing 'posterior_beat_prob' column")
    print(f"⚠️ Insufficient data for quad-model agreement ({'; '.join(reasons)})")

⚠️ Insufficient data for quad-model agreement (beat_results is empty)


### 7.5 Cross-Model Dependency Structure (Gaussian Copula)


In [24]:
# Measure tail dependence between MC and Kalman return signals
if len(mc_kal) > 50:
    copula_result = fit_gaussian_copula(
        mc_kal,
        features=['expected_upside_pct', 'filtered_upside']
    )
    if copula_result:
        print(f"📊 MC ↔ Kalman Dependency Analysis:")
        print(f"   Correlation matrix:\n{copula_result.get('correlation_matrix', 'N/A')}")
        print(f"   Tail dependence: {copula_result.get('tail_dependence', 'N/A')}")


📊 MC ↔ Kalman Dependency Analysis:
   Correlation matrix:
[[1.         0.94922202]
 [0.94922202 1.        ]]
   Tail dependence: {'lower': array([[1.        , 0.78358209],
       [0.78358209, 1.        ]]), 'upper': array([[1.        , 0.84328358],
       [0.84328358, 1.        ]])}


## 8. Sector Expected Returns Heatmap


In [25]:
# Aggregate all return metrics by sector
sector_returns = (
    tri.groupby("industry")
    .agg(
        mc_mean=("expected_upside_pct", "mean"),
        mc_median=("expected_upside_pct", "median"),
        kalman_mean=("filtered_upside", "mean"),
        kalman_median=("filtered_upside", "median"),
        pt_mean=("expected_return_prob_weighted", "mean"),
        pt_median=("expected_return_prob_weighted", "median"),
        pct_bullish=("agreement_score", lambda x: (x == 3).mean() * 100),
        count=("ticker", "count"),
    )
    .reset_index()
)

heatmap_data = sector_returns.set_index("industry")[
    ["mc_mean", "mc_median", "kalman_mean", "kalman_median", "pt_mean", "pt_median", "pct_bullish"]
].rename(columns={
    "mc_mean": "MC Mean",
    "mc_median": "MC Median",
    "kalman_mean": "Kalman Mean",
    "kalman_median": "Kalman Median",
    "pt_mean": "Achiev. Mean",
    "pt_median": "Achiev. Median",
    "pct_bullish": "% All Bullish",
})

fig = px.imshow(
    heatmap_data.round(1),
    color_continuous_scale="RdYlGn",
    text_auto=True,
    aspect="auto",
    title="Industry Expected Returns Heatmap (All Models)",
    labels={"color": "Value"},
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    height=1250,
)
fig.show()


## 9. VaR & Tail Risk Analysis


In [26]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("VaR 5% Distribution", "VaR 5% vs Expected Upside"),
    vertical_spacing=0.12,
)

# VaR distribution
var_clipped = mc["var_5_pct"].clip(-150, 300)
fig.add_trace(
    go.Histogram(
        x=var_clipped,
        nbinsx=80,
        marker_color=COLORS[3],
        opacity=0.75,
        name="VaR 5%",
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="blue", row=1, col=1)

# VaR vs Expected Upside (sampled for performance)
sample = mc.sample(min(2000, len(mc)), random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample["var_5_pct"],
        y=sample["expected_upside_pct"],
        mode="markers",
        marker=dict(
            size=4,
            color=sample["prob_positive_upside"],
            colorscale="RdYlGn",
            colorbar=dict(title="P(+)"),
            opacity=0.5,
        ),
        text=sample["name"],
        hovertemplate="%{text}<br>VaR 5%%: %{x:.1f}%<br>Expected Upside: %{y:.1f}%<extra></extra>",
        name="Stocks",
    ),
    row=2, col=1,
)
fig.add_shape(
    type="line", x0=-100, y0=-100, x1=300, y1=300,
    line=dict(color="gray", dash="dash", width=1),
    row=2, col=1,
)

fig.update_layout(
    title="Value-at-Risk (5%) Analysis",
    template=PLOTLY_TEMPLATE,
    height=800,
    width=1000,
    showlegend=False,
)
fig.update_xaxes(title_text="VaR 5% (%)", row=1, col=1)
fig.update_xaxes(title_text="VaR 5% (%)", row=2, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Expected Upside (%)", row=2, col=1)
fig.show()


In [27]:
identifier_cols = load_identifier_columns()

def _reorder_with_identifiers(result_df: pd.DataFrame) -> pd.DataFrame:
    id_cols = [c for c in identifier_cols if c in result_df.columns]
    other_cols = [c for c in result_df.columns if c not in id_cols]
    return result_df[id_cols + other_cols]


## 9.5b Portfolio Monte Carlo Simulation (Top Consensus Picks)


In [28]:
from finance_ml.ml_workflow.analytics.risk import run_monte_carlo_simulation

if len(strong_consensus) >= 4:
    # Build equal-weight portfolio from strong consensus picks
    n_picks = min(20, len(strong_consensus))
    weights = np.ones(n_picks) / n_picks
    
    # This would require daily returns data — placeholder for integration
    print(f"📊 Portfolio MC ready for {n_picks} strong consensus picks")
    print("   Requires daily returns DataFrame — integrate with market data feed")


📊 Portfolio MC ready for 20 strong consensus picks
   Requires daily returns DataFrame — integrate with market data feed


## 9.5c Inline Fast MC Simulation (Numba-accelerated)


In [29]:
# Optional: Inline fast MC simulation (Numba-accelerated)
opt_status = get_optimization_status()
print(f"🔧 Numba available: {opt_status.get('numba_available')}")

# mc_inline = fast_monte_carlo_simulation(source_df, n_simulations=10000)


🔧 Numba available: False


## 9.5 InferenceData Schema Integration

Build ArviZ-compatible InferenceData from Monte Carlo simulation results
for standardised posterior analysis, diagnostics, and NetCDF export.


In [30]:
# Build InferenceData from Monte Carlo simulation results
if ARVIZ_AVAILABLE and 'mc' in dir() and len(mc) > 0:
    try:
        idata_mc = build_monte_carlo_inference_data(
            mc, mc, n_simulations=10000,
        )
        mc_summary = summarize_inference_data(idata_mc)
        print(f"✅ InferenceData built: {mc_summary.get('groups', [])}")
        print(f"   Draws: {mc_summary.get('n_draws', 0)}, Equities: {mc_summary.get('n_equities', 0)}")
        if mc_summary.get('r_hat'):
            for var, rhat_val in mc_summary['r_hat'].items():
                print(f"   R-hat ({var}): {rhat_val:.4f}")
    except Exception as e:
        print(f"⚠️ InferenceData build failed: {e}")
else:
    print('⚠️ ArviZ not available or no MC data')


✅ InferenceData built: ['posterior_predictive', 'observed_data', 'constant_data']
   Draws: 10000, Equities: 5378


## 9.6 Probabilistic Visualizations

Generate ArviZ-backed probabilistic charts from Monte Carlo and Bayesian results.


In [31]:
# Posterior return forest from Monte Carlo results
if 'mc' in dir() and len(mc) > 0:
    fig = create_posterior_return_forest(mc, top_n=25)
    fig.show()
    fig.write_html('outputs/analytics/posterior_return_forest.html')
    print('✓ Saved posterior_return_forest.html')


✓ Saved posterior_return_forest.html


In [32]:
# Tri-model posterior comparison (requires tri-model alignment DataFrame)
tri_cols = {'name', 'expected_upside_pct', 'filtered_upside', 'expected_return_prob_weighted'}
if 'strong_consensus' in dir() and tri_cols.issubset(strong_consensus.columns):
    fig = create_tri_model_posterior_comparison(strong_consensus, top_n=25)
    fig.show()
    fig.write_html('outputs/analytics/tri_model_posterior_comparison.html')
    print('✓ Saved tri_model_posterior_comparison.html')
else:
    print('⚠️ Tri-model columns not available — skipping')


✓ Saved tri_model_posterior_comparison.html


## 10. Summary Statistics


In [33]:
summary = {
    "Monte Carlo": {
        "Stocks Analyzed": len(mc),
        "Mean Expected Upside (%)": mc["expected_upside_pct"].mean().round(2),
        "Median Expected Upside (%)": mc["expected_upside_pct"].median().round(2),
        "% Stocks with Positive Upside": (mc["expected_upside_pct"] > 0).mean() * 100,
        "Mean Prob Positive (%)": mc["prob_positive_upside"].mean().round(1),
    },
    "Price Target Achievement": {
        "Stocks Analyzed": len(pt),
        "Mean Prob-Weighted Return (%)": pt["expected_return_prob_weighted"].mean().round(2),
        "Mean Achievement Prob": pt["achievement_probability"].mean().round(3),
        "High Confidence Count": (pt["confidence_level"] == "High").sum(),
        "Mean Analyst Conviction (%)": pt["analyst_conviction"].mean().round(1),
    },
    "Kalman Filter": {
        "Stocks Analyzed": len(kal),
        "Mean Filtered Upside (%)": kal["filtered_upside"].mean().round(2),
        "Median Filtered Upside (%)": kal["filtered_upside"].median().round(2),
        "Mean Signal Strength": kal["signal_strength"].mean().round(2),
        "% Positive Filtered Upside": (kal["filtered_upside"] > 0).mean() * 100,
    },
}

summary_df = pd.DataFrame(summary).T
display(summary_df)

if len(tri) > 0:
    print(f"\n🔗 Cross-Model Coverage: {len(tri):,} stocks in all 3 models")
    print(
        f"   Strong Bullish (3/3 agree): {(tri['agreement_score'] == 3).sum():,} ({(tri['agreement_score'] == 3).mean() * 100:.1f}%)")
    print(
        f"   Strong Bearish (0/3 agree): {(tri['agreement_score'] == 0).sum():,} ({(tri['agreement_score'] == 0).mean() * 100:.1f}%)")
    print(f"   MC ↔ Kalman correlation:    {tri[['expected_upside_pct', 'filtered_upside']].corr().iloc[0, 1]:.3f}")
    print(
        f"   MC ↔ Achievement corr:      {tri[['expected_upside_pct', 'expected_return_prob_weighted']].corr().iloc[0, 1]:.3f}")

print("\n✅ Expected Returns Analytics complete")


,Stocks Analyzed,Mean Expected Upside (%),Median Expected Upside (%),% Stocks with Positive Upside,Mean Prob Positive (%),Mean Prob-Weighted Return (%),Mean Achievement Prob,High Confidence Count,Mean Analyst Conviction (%),Mean Filtered Upside (%),Median Filtered Upside (%),Mean Signal Strength,% Positive Filtered Upside
Monte Carlo,5378.0,22.89,13.84,76.943102,75.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price Target Achievement,6174.0,NaN,NaN,NaN,NaN,8.52,0.605,964.0,62.3,NaN,NaN,NaN,NaN
Kalman Filter,6171.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.57,12.8,11.0,77.134986



🔗 Cross-Model Coverage: 5,378 stocks in all 3 models
   Strong Bullish (3/3 agree): 3,958 (73.6%)
   Strong Bearish (0/3 agree): 991 (18.4%)
   MC ↔ Kalman correlation:    0.922
   MC ↔ Achievement corr:      0.739

✅ Expected Returns Analytics complete


## 11. Export Results to Analytics Database


In [34]:
# Export tri-model consensus results
if len(tri) > 0:
    export_to_analytics_db(
        _reorder_with_identifiers(tri),
        "expected_returns_tri_model"
    )
    print(f"✓ Exported {len(tri)} rows → analytics.expected_returns_tri_model")

# Export strong consensus picks
if len(strong_consensus) > 0:
    export_to_analytics_db(
        _reorder_with_identifiers(strong_consensus),
        "strong_consensus_picks"
    )
    print(f"✓ Exported {len(strong_consensus)} rows → analytics.strong_consensus_picks")


✓ Exported 5378 rows → analytics.expected_returns_tri_model
✓ Exported 50 rows → analytics.strong_consensus_picks
